# 01 – Neural Networks with TensorFlow / Keras

Topics covered:
1. Perceptron & feedforward network intuition
2. Building a Dense network with Keras
3. Activation functions, optimisers, loss functions
4. Callbacks: EarlyStopping, ModelCheckpoint, LearningRateScheduler
5. Regularisation: Dropout, L2
6. Training curves & overfitting

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.datasets import load_breast_cancer, fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, r2_score

print('TensorFlow version:', tf.__version__)
tf.random.set_seed(42)
np.random.seed(42)

## 1. Binary Classification – Breast Cancer

In [ ]:
bc = load_breast_cancer()
X, y = bc.data, bc.target.astype(np.float32)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_tr).astype(np.float32)
X_te_sc  = scaler.transform(X_te).astype(np.float32)

In [ ]:
# Build model
def build_classifier(input_dim, dropout_rate=0.3):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        layers.BatchNormalization(),
        layers.Dropout(dropout_rate),
        layers.Dense(32, activation='relu'),
        layers.Dropout(dropout_rate),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

clf = build_classifier(X_tr_sc.shape[1])
clf.summary()

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6)
]

history = clf.fit(
    X_tr_sc, y_tr,
    validation_split=0.15,
    epochs=200,
    batch_size=32,
    callbacks=callbacks,
    verbose=0
)

print('Test Accuracy:', clf.evaluate(X_te_sc, y_te, verbose=0)[1]:.3f)

In [ ]:
# Training curves
def plot_history(hist, metrics=['loss', 'accuracy']):
    fig, axes = plt.subplots(1, len(metrics), figsize=(6 * len(metrics), 4))
    if len(metrics) == 1:
        axes = [axes]
    for ax, m in zip(axes, metrics):
        ax.plot(hist.history[m],         label='Train')
        ax.plot(hist.history[f'val_{m}'], label='Val')
        ax.set_xlabel('Epoch'); ax.set_ylabel(m.capitalize())
        ax.set_title(f'{m.capitalize()} Curve')
        ax.legend()
    plt.tight_layout(); plt.show()

plot_history(history)

## 2. Multi-class Classification – Fashion MNIST

In [ ]:
(X_train, y_train), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()

X_train = X_train.reshape(-1, 784).astype('float32') / 255.0
X_test  = X_test.reshape(-1, 784).astype('float32') / 255.0

class_names = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print('Train:', X_train.shape, '  Test:', X_test.shape)

In [ ]:
mc_model = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(10, activation='softmax')
])

mc_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

hist_mc = mc_model.fit(
    X_train, y_train,
    validation_split=0.1,
    epochs=30,
    batch_size=128,
    callbacks=[EarlyStopping(patience=5, restore_best_weights=True)],
    verbose=1
)

test_loss, test_acc = mc_model.evaluate(X_test, y_test, verbose=0)
print(f'\nTest Accuracy: {test_acc:.3f}')

In [ ]:
plot_history(hist_mc)

## 3. Regression – California Housing

In [ ]:
housing = fetch_california_housing(as_frame=True)
X_h = housing.data.values.astype(np.float32)
y_h = housing.target.values.astype(np.float32)

X_h_tr, X_h_te, y_h_tr, y_h_te = train_test_split(X_h, y_h, test_size=0.2, random_state=42)
sc_h = StandardScaler()
X_h_tr_sc = sc_h.fit_transform(X_h_tr).astype(np.float32)
X_h_te_sc  = sc_h.transform(X_h_te).astype(np.float32)

reg_model = keras.Sequential([
    layers.Input(shape=(X_h_tr_sc.shape[1],)),
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(1)  # linear output for regression
])
reg_model.compile(optimizer='adam', loss='mse', metrics=['mae'])

hist_reg = reg_model.fit(
    X_h_tr_sc, y_h_tr,
    validation_split=0.1,
    epochs=100,
    batch_size=64,
    callbacks=[EarlyStopping(patience=10, restore_best_weights=True)],
    verbose=0
)

y_h_pred = reg_model.predict(X_h_te_sc).ravel()
print(f'MAE : {np.mean(np.abs(y_h_te - y_h_pred)):.3f}')
print(f'R²  : {r2_score(y_h_te, y_h_pred):.3f}')

## 4. Key Takeaways

| Concept | Notes |
|---------|-------|
| Layers | Input → Dense(ReLU) → … → Output |
| Activation | ReLU (hidden), sigmoid (binary), softmax (multi-class) |
| Loss | binary_crossentropy / sparse_categorical_crossentropy / mse |
| BatchNorm | Normalises activations; stabilises training |
| Dropout | Randomly zeros neurons; prevents overfitting |
| EarlyStopping | Stop when val_loss stops improving |

**Next:** `02_CNN_Image_Classification.ipynb`